# 00 — Baseline com modelos clássicos (SGDClassifier e RandomForestClassifier)

Antes de partir para a CNN (`02_cnn.ipynb`), este notebook treina os **modelos mínimos
exigidos** para tarefas de classificação: `SGDClassifier` e `RandomForestClassifier`
(Scikit-Learn), seguindo a mesma referência do Capítulo 3 do livro-texto (classificação
de dígitos do MNIST).

Diferente da CNN, esses modelos **não enxergam a estrutura espacial da imagem**: cada
imagem 28×28 é achatada em um vetor de **784 atributos** (um por pixel) e tratada como um
problema de classificação tabular comum.

**Metodologia (mesma disciplina dos outros notebooks):**
- Divisão 50k treino / 10k validação / 10k teste, com **estratificação** por classe.
- A validação escolhe entre os dois baselines; nenhuma decisão é tomada olhando o teste.
- O **teste é usado uma única vez**, ao final, para uma estimativa honesta de desempenho.

In [ ]:
# ── Setup — funciona igual no seu PC e no Google Colab ───────────────────────
# Ajuste REPO_URL após publicar o projeto no GitHub (ver README).
import sys, os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/4rth-g/fashion-mnist-fundamentos-ia.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # No Colab não há o repositório: clonamos para ter o código de src/ e as pastas.
    if not Path("fashion-mnist-fundamentos-ia").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("fashion-mnist-fundamentos-ia")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seaborn"], check=True)

# Torna o pacote src/ importável (local: rodando de notebooks/; Colab: da raiz).
_root = Path.cwd()
_src = _root / "src" if (_root / "src").exists() else _root.parent / "src"
sys.path.insert(0, str(_src))
print("src/ em:", _src)

In [ ]:
import time
import json
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score, accuracy_score)

from utils import seed_everything, load_fashion_mnist, CLASS_NAMES, FIGURES_DIR, RESULTS_DIR, SEED

seed_everything()

## Preparando os dados para modelos clássicos

`SGDClassifier` e `RandomForestClassifier` esperam uma matriz 2D `(n_amostras, n_atributos)`,
não um tensor de imagem. Por isso, o pré-processamento aqui é diferente do usado na CNN:

1. **Escala dos pixels:** carregamos as imagens com `normalize=False`, mantendo os pixels em
   `[0, 1]` (já convertidos de `[0, 255]` pelo `ToTensor`). Essa escala já é adequada para os
   dois modelos — o `RandomForestClassifier` é invariante à escala dos atributos (decide por
   splits, não por distância), e o `SGDClassifier` já trabalha bem com atributos num intervalo
   pequeno e comparável entre si. Por isso **não aplicamos um `StandardScaler` adicional** aqui
   (diferente do que faríamos com dados tabulares heterogêneos, como no projeto do Capítulo 2).
2. **Achatamento (`flatten`):** cada imagem 28×28 vira um vetor de 784 atributos.
3. **Separação treino/validação estratificada:** os 60k de treino são divididos em **50k
   treino / 10k validação**, preservando a proporção das 10 classes em cada subconjunto
   (`stratify=y`) — mais rigoroso que uma divisão puramente aleatória.
4. O conjunto de **teste (10k) permanece intocado** até a avaliação final.

5. **Valores extremos e atributos irrelevantes:** já investigados em
   `01_eda.ipynb` — só 0,06% das imagens têm intensidade atípica (não removidas,
   são peças válidas) e apenas 1,4% dos 784 pixels (concentrados nos cantos) têm
   variância praticamente nula entre as imagens. Não removemos esses atributos:
   é uma fração pequena demais para valer a complexidade extra, e o
   `RandomForestClassifier` naturalmente ignora atributos pouco informativos ao
   escolher os splits das árvores.

> A conversão para arrays NumPy pode levar até 1 minuto (percorre as ~70.000 imagens).

In [ ]:
train_full, test = load_fashion_mnist(normalize=False)


def dataset_to_arrays(dataset):
    """Converte um torchvision Dataset em (X, y) NumPy: X achatado em (n, 784)."""
    X = np.stack([img.numpy().reshape(-1) for img, _ in dataset]).astype(np.float32)
    y = dataset.targets.numpy()
    return X, y


t0 = time.time()
X_train_full, y_train_full = dataset_to_arrays(train_full)
X_test, y_test = dataset_to_arrays(test)
print(f"Conversão para arrays concluída em {time.time() - t0:.1f}s")
print(f"X_train_full: {X_train_full.shape} | X_test: {X_test.shape}")

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=10_000, stratify=y_train_full, random_state=SEED,
)
print(f"Treino: {X_train.shape} | Validação: {X_val.shape}")

# Confere que a estratificação preservou o balanceamento das classes.
print("\nProporção de classes (treino):    ", np.round(np.bincount(y_train) / len(y_train), 3))
print("Proporção de classes (validação): ", np.round(np.bincount(y_val) / len(y_val), 3))

## Baseline 1 — SGDClassifier

O `SGDClassifier` treina um classificador linear via gradiente descendente estocástico —
rápido mesmo em datasets grandes, mas incapaz de capturar relações não lineares entre pixels
(ele não "enxerga" que pixels vizinhos formam bordas ou texturas, ao contrário da CNN).

In [ ]:
sgd_clf = SGDClassifier(random_state=SEED, max_iter=1000, tol=1e-3, n_jobs=-1)

t0 = time.time()
sgd_clf.fit(X_train, y_train)
print(f"SGDClassifier treinado em {time.time() - t0:.1f}s")

sgd_val_pred = sgd_clf.predict(X_val)
sgd_val_acc = accuracy_score(y_val, sgd_val_pred)
print(f"Acurácia na validação: {sgd_val_acc:.4f}")

## Baseline 2 — RandomForestClassifier

O `RandomForestClassifier` combina várias árvores de decisão treinadas em subconjuntos
aleatórios dos dados e dos atributos. Cada árvore consegue aprender combinações simples e não
lineares entre pixels (ex.: "se o pixel X é claro e o pixel Y é escuro..."), mas ainda sem
noção de vizinhança espacial — algo que só a convolução da CNN explora de fato.

> Pode levar de 1 a 3 minutos na CPU (100 árvores sobre 50.000 imagens de 784 atributos).

In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)

t0 = time.time()
rf_clf.fit(X_train, y_train)
print(f"RandomForestClassifier treinado em {time.time() - t0:.1f}s")

rf_val_pred = rf_clf.predict(X_val)
rf_val_acc = accuracy_score(y_val, rf_val_pred)
print(f"Acurácia na validação: {rf_val_acc:.4f}")

## Comparando os dois baselines (na validação)

Assim como na CNN, usamos a **validação** — nunca o teste — para decidir qual dos dois
baselines é melhor antes de medir o desempenho final.

In [ ]:
comparison = pd.DataFrame({
    "modelo": ["SGDClassifier", "RandomForestClassifier"],
    "acuracia_validacao": [sgd_val_acc, rf_val_acc],
}).sort_values("acuracia_validacao", ascending=False).reset_index(drop=True)
print(comparison)

best_name = comparison.iloc[0]["modelo"]
best_model = sgd_clf if best_name == "SGDClassifier" else rf_clf
print(f"\nMelhor baseline pela validação: {best_name}")

## Avaliação final no conjunto de TESTE (uma única vez)

Assim como em `02_cnn.ipynb`, o teste só é tocado agora, com o baseline vencedor já escolhido
pela validação — garantindo uma estimativa honesta de desempenho em dados nunca vistos.

In [ ]:
test_pred = best_model.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)
macro_f1 = f1_score(y_test, test_pred, average="macro")

print(f"[{best_name}] Acurácia no TESTE: {test_acc:.4f} | F1 macro: {macro_f1:.4f}\n")
print(classification_report(y_test, test_pred, digits=4, target_names=CLASS_NAMES))

## Matriz de confusão do baseline vencedor

In [ ]:
cm = confusion_matrix(y_test, test_pred)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=ax, cmap="Oranges", colorbar=False, xticks_rotation=45)
ax.set_title(f"Matriz de confusão — {best_name} (teste, acc {test_acc:.4f})")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "baseline_matriz_confusao.png", dpi=120)
plt.show()

## Baseline vs. CNN

Como referência, comparamos aqui o melhor baseline clássico com a CNN treinada em
`02_cnn.ipynb` (lida de `results/metrics.json`, se esse notebook já tiver sido executado
antes deste).

In [ ]:
cnn_metrics_path = RESULTS_DIR / "metrics.json"
cnn_acc = None
if cnn_metrics_path.exists():
    with open(cnn_metrics_path, encoding="utf-8") as f:
        cnn_acc = json.load(f).get("test_accuracy")

bar_labels = [f"Baseline\n({best_name})", "CNN\n(02_cnn.ipynb)"]
bar_values = [test_acc, cnn_acc if cnn_acc is not None else np.nan]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(bar_labels, bar_values, color=["darkorange", "steelblue"])
ax.set_ylim(0, 1)
ax.set_ylabel("Acurácia no teste")
ax.set_title("Baseline clássico vs. CNN — acurácia no teste")
for bar, val in zip(bars, bar_values):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f"{val:.4f}", ha="center")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "baseline_vs_cnn.png", dpi=120)
plt.show()

if cnn_acc is None:
    print("Aviso: results/metrics.json não encontrado — rode 02_cnn.ipynb ao menos uma vez "
          "para incluir a CNN nesta comparação.")
else:
    diff = cnn_acc - test_acc
    print(f"A CNN supera o melhor baseline ({best_name}) em {diff:.4f} "
          f"({diff * 100:.2f} pontos percentuais) de acurácia no teste.")

In [ ]:
baseline_metrics = {
    "sgd_val_accuracy": float(sgd_val_acc),
    "random_forest_val_accuracy": float(rf_val_acc),
    "best_baseline": best_name,
    "test_accuracy": float(test_acc),
    "macro_f1": float(macro_f1),
    "per_class": classification_report(y_test, test_pred, target_names=CLASS_NAMES,
                                       output_dict=True),
    "confusion_matrix": cm.tolist(),
    "class_names": CLASS_NAMES,  # ordem das linhas/colunas de confusion_matrix
    "gerado_em": datetime.now().isoformat(timespec="seconds"),
}
with open(RESULTS_DIR / "baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=2, ensure_ascii=False)
print(f"Métricas do baseline salvas em {RESULTS_DIR / 'baseline_metrics.json'}")

## Conclusão

- Os dois modelos mínimos exigidos (`SGDClassifier`, `RandomForestClassifier`) foram treinados
  e avaliados com a mesma disciplina metodológica do restante do projeto (validação escolhe o
  modelo, teste avalia uma única vez).
- O **RandomForestClassifier** tende a superar o `SGDClassifier` nesta tarefa, já que consegue
  combinar não linearidades simples entre pixels — mas nenhum dos dois enxerga a estrutura
  espacial da imagem como a CNN.
- O **baseline serve de patamar de comparação**: se a CNN (`02_cnn.ipynb`) não superasse estes
  modelos mais simples, isso seria um sinal de que a complexidade extra não estaria se
  traduzindo em benefício real.
- **Limitação dos baselines:** como cada pixel é tratado como um atributo independente, um leve
  deslocamento da peça na imagem já é suficiente para confundi-los — a CNN, por combinar
  padrões locais (bordas, texturas) via convolução, é bem mais robusta a esse tipo de variação.